# Outline

- Preparing Workspace

Importing packages, defining file paths, running user defined functions, setting API key, ...

- Preparing Imports

This section imports the "Census Configuration File.xlsx" and sets the user defined inputs to objects that are used throughout the script

- Importing
- Processing
- Exporting

***

## Preparing Workspace

***

In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

In [9]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

<>:8: SyntaxWarning: invalid escape sequence '\R'
<>:8: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_34024\1210751357.py:8: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')


In [3]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

## Preparing Imports

***

In [13]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying Census data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]

# View
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import about table
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
MOE_thresh = df_about['MOE Threshold'].values[0]
print(folder)
print('MOE threshold: ' + str(MOE_thresh) + '%')

Health_1
CPS
FOODSEC
Counties
Counties
Percentages: No
Margin of error: No
Number of variables: 4
2009
2022
Access to Grocery Stores
MOE threshold: Not sure if I can pull SE's yet%


In [14]:
## Import Variable Mapping
df_inputs = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = import_tab)

# Set years
years_to_import = list(range(year_start, year_end+1))

## For DEC data
if estimate == 'DEC':

    # Reset years to import for DEC
    # Set DEC variables to import
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = estimate)
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    if margin_of_error == 'Yes':
        df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)

    years_to_import = [2000, 2010, 2020]
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['NAME'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list())
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    # view
    print(dict_fips)
    print(dict_vars)

## For ACS1 or ACS5 data
if sample_type in ['ACS', 'SUBJECT']:

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    # Remove 2020 if pulling ACS1 tables (Census did not take an ACS1 sample in 2020)
    # Set tables and variables to import    

    if estimate == 'ACS1':
        try:
            years_to_import.remove(2020)
        except Exception as e: print(e)

    if margin_of_error == 'Yes':
        df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)
        list_vars = ['NAME'] + df_vars['ID_Attributes'].to_list()
    else:
        list_vars = ['NAME'] + df_vars['ID'].to_list()

    if sample_type == 'ACS':
        tables = df_vars['Table'].unique()
        print(tables)

    # For tract and county level pull
    if import_tab == 'Counties':
        
        # Import County FIPS mapping
        # Convert to dictionary object for easy state-county combination importing
        df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        df_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values))
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = df_fips.copy()
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(list_vars)

    
    # For MSA level pull
    if import_tab == 'MSA':
    
        # Set MSAs to import
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)
        print(list_vars)


## For PUMS data
if sample_type == 'PUMS':

    # Remove 2012-2015 if pulling PUMS tables (they only reported at the state level for PUMS on these years)
    # Create dictionary of variable mappings by year (sometimes the variable name changes over time)
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    table_type = df_inputs['table'].values[0]
    print('PUMS table roll up: ' + table_type)

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]

    if margin_of_error == 'Yes':
        df_vars.loc[(df_vars['ID'].str.contains('WGTP')) & (df_vars['Table Type'] == table_type), 'Include'] = 'Yes'
    
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    if table_type == 'P':
        weight = 'PWGTP'
    if table_type == 'H':
        weight = 'WGTP'
    groups  = list(df_vars[df_vars['Data Type'].str.contains('group')]['ID2'].unique())
    groups2 = list(df_vars[df_vars['Data Type'] == 'group']['ID2'].unique())
    print(groups)
    print(groups2)
    
    if estimate == 'ACS5':
        try:
            years_to_import.remove(2012)
            years_to_import.remove(2013)
            years_to_import.remove(2014)
            years_to_import.remove(2015)
        except Exception as e: print(e)

    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
            
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[(df_vars['Year'] == year) & (df_vars['Data Type'].str.contains('group'))]['ID'].to_list()) + unique(df_vars[(df_vars['Year'] == year) & (df_vars['Data Type'] == 'integer')]['ID'].to_list()) + [weight]
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                 , sheet_name = 'PUMAcodes'
                                 , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS'})

    df_fips = df_fips.merge(df_fips_pums[['State FIPS', 'County FIPS', 'PUMA5CE']].drop_duplicates(), on = ['State FIPS', 'County FIPS'])
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'PUMA5CE']].drop_duplicates()
    
    dict_fips = dict_fips.groupby('State FIPS')['PUMA5CE'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])

    # view
    print(dict_fips)
    print(dict_vars)



if estimate == 'CPS':
    
    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
    
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    
    df_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = df_fips.copy()
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
        
    # view
    print(dict_fips)
    print(dict_vars)


# view
df_vars.head(3)

C:\Users\jchoy\AppData\Local\Temp\ipykernel_34024\1584488560.py:181: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]


{'06': '017,061,067,101,113,115'}
{'2009': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2010': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2011': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2012': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2013': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2014': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2015': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2016': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2017': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2018': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON', 'PERRP', 'PTDTRACE', 'HHSUPWGT'], '2019': ['HES1A', 'HES1B', 'HES1C', 'HES1D', 'PEHSPNON'

,Label,ID,Value1,Value2,Description,Suggested Weight,Year,Indicator Name,ID2,Description2,Include,Data Type
440,Expend shopped at supermarket/grocery store la...,HES1A,-1,-1,Not in Universe,HHSUPWGT,2022,Health_1,HES1A,Not in Universe,Yes,group
441,Expend shopped at supermarket/grocery store la...,HES1A,-2,-2,Don't Know,HHSUPWGT,2022,Health_1,HES1A,Don't Know,Yes,group
442,Expend shopped at supermarket/grocery store la...,HES1A,-3,-3,Refused,HHSUPWGT,2022,Health_1,HES1A,Refused,Yes,group


***

## Importing

***

In [15]:
start_time = time.time()

# Import Census Bureau data to url mapping
df_urls = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'URL')

## For ACS tables
if sample_type == 'ACS':
  
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling ACS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # reduce all tables/variables pulled into one table

    list_df_census = []
    
    for table in tables:
    
        print("")
        print("Table ID: " + table)
        print("")
        list_df_tables = []
    
        df_table = df_vars[df_vars['Table'] == table]
        if margin_of_error == 'Yes':
            list_table_vars = [['NAME'] + df_table['ID_Attributes'].to_list()[x:x+20] for x in range(0, len(df_table['ID_Attributes'].to_list()), 20)]
        else:
            list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+45] for x in range(0, len(df_table['ID'].to_list()), 45)]
        
        list_variables = []
        for x in list_table_vars:
            list_variables.append(",".join(x))
        
        list_df_vars = []
        
        for variables in list_variables:
            print("Variables: " + variables)
            list_df_years = []

            if import_tab == 'Counties':
                for state in list(dict_fips.keys()):
                    print('State: ' + state)
                    for year in tqdm(years_to_import):
                        try:
                            list_df_years.append(
                                query_census(df_urls      = df_urls
                                              , api_key   = api_key
                                              , estimate  = estimate
                                              , sample    = sample_type
                                              , geography = geography
                                              , variables = variables
                                              , year      = year
                                              , state     = state
                                              , county    = dict_fips[state])
                            )
                        except Exception as e: print(e)
                df_years = pd.concat(list_df_years)
                            
            if import_tab == 'MSA':
                for year in tqdm(years_to_import):
                    try:
                        list_df_years.append(
                            query_census(df_urls      = df_urls
                                          , api_key   = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , msa       = msa_to_import)
                        )
                    except Exception as e: print(e)
                df_years = pd.concat(list_df_years)

            list_df_vars.append(df_years)
        
        if geography == 'Tracts':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'Counties':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'MSA':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'outer'), list_df_vars)

        list_df_census.append(df_vars_all)
        print("All variables from table ID " + table + " have been reduced together into one table")
        print("")

    print("")
    print("Reducing all tables together into one final table...")
    print("")
    
    if geography == 'Tracts':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()
    if geography == 'MSA':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']), list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']).reset_index()



## For SUBJECT tables

if sample_type == 'SUBJECT':
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling ACS Subject data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # reduce all tables/variables pulled into one table

    list_df_census = []
    if margin_of_error == 'Yes':
        list_table_vars = [['NAME'] + df_vars['ID_Attributes'].to_list()[x:x+20] for x in range(0, len(df_vars['ID_Attributes'].to_list()), 20)]
    else:
        list_table_vars = [['NAME'] + df_vars['ID'].to_list()[x:x+45] for x in range(0, len(df_vars['ID'].to_list()), 45)]
    
    list_variables = []
    for x in list_table_vars:
        list_variables.append(",".join(x))
    
    list_df_vars = []
    
    for variables in list_variables:
        print("Variables: " + variables)
        list_df_years = []
        if import_tab == 'Counties':
            for state in list(dict_fips.keys()):
                print('State: ' + state)
                for year in tqdm(years_to_import):
                    try:
                        list_df_years.append(
                            query_census(df_urls      = df_urls
                                          , api_key   = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , state     = state
                                          , county    = dict_fips[state])
                        )
                    except Exception as e: print(e)
            df_years = pd.concat(list_df_years)
                        
        if import_tab == 'MSA':
            for year in tqdm(years_to_import):
                try:
                    list_df_years.append(
                        query_census(df_urls      = df_urls
                                      , api_key   = api_key
                                      , estimate  = estimate
                                      , sample    = sample_type
                                      , geography = geography
                                      , variables = variables
                                      , year      = year
                                      , msa       = msa_to_import)
                    )
                except Exception as e: print(e)
            df_years = pd.concat(list_df_years)
        list_df_vars.append(df_years)
    
    if geography == 'Tracts':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'outer'), list_df_vars)
    if geography == 'Counties':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year'], how = 'outer'), list_df_vars)
    if geography == 'MSA':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'outer'), list_df_vars)
    list_df_census.append(df_vars_all)

    print("")
    print("Reducing all tables together into one final table...")
    print("")
    
    if geography == 'Tracts':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()
    if geography == 'MSA':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']), list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']).reset_index()



## For DEC tables
if estimate == 'DEC':

    print("Importing and compiling Decennial data from the Census Bureau...")
    print("")

    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties
    # reduce all tables/variables pulled into one table
    
    list_df_census = []
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                list_df_census.append(
                    query_census(df_urls       = df_urls
                                   , api_key   = api_key
                                   , estimate  = estimate
                                   , sample    = sample_type
                                   , geography = geography
                                   , variables = ','.join(dict_vars[str(year)])
                                   , year      = year
                                   , state     = state
                                   , county    = dict_fips[state])
                )
            except Exception as e: print(e)
                    
    if geography == 'Tracts':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()


## For PUMS tables
if geography == 'PUMA':
    
    print("Importing and compiling PUMS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and PUMAs
    # import all variables
    # combine all years and PUMAs
    # outer join variables onto ID fields for each geography type
    # calculate margin of error using replicate weights
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        list_df_years = []
        for year in years_to_import:
            print("")
            print('Year: ' + str(year))
            list_df_vars = []
            try:
                list_table_vars = [dict_vars[str(year)][x:x+45] for x in range(0, len(dict_vars[str(year)]), 45)]
                
                list_variables = []
                for x in list_table_vars:
                    list_variables.append(",".join(x))

                print('Querying variables...')
                for variables in tqdm(list_variables):
                    df_pums = query_census(df_urls       = df_urls
                                             , api_key   = api_key
                                             , estimate  = estimate
                                             , sample    = sample_type
                                             , geography = geography
                                             , variables = 'PUMA,SERIALNO,SPORDER,' + variables
                                             , year      = year
                                             , state     = state
                                             , puma      = dict_fips[state])
                    df_pums['state'] = state
                    df_pums = df_pums.drop(['public use microdata area'], axis = 1)
                    list_df_vars.append(df_pums)

                df_vars_years = ft.reduce(lambda left, right: pd.merge(left, right, on = ['state', 'SERIALNO', 'Year', 'PUMA', 'SPORDER'], how = 'left'), list_df_vars)
                df_vars_years = df_vars_years.set_index(['state', 'SERIALNO', 'Year', 'PUMA', 'SPORDER']).reset_index()
                df_vars_years.columns = ['state', 'SERIALNO', 'Year', 'PUMA', 'SPORDER'] + dict_vars[str(np.max(years_to_import))]
                if (table_type == 'H') & ('SPORDER' in df_vars_years.columns):
                    df_vars_years = df_vars_years[df_vars_years['SPORDER'] == '1']
                    df_vars_years = df_vars_years.drop('SPORDER', axis = 1)
                if margin_of_error == 'Yes':
                    print('Calculating margin of error using replicate weights...')
                    cols = [col for col in df_vars_years.columns if weight in col]
                    df_vars_years[cols] = df_vars_years[cols].astype(int)
                    cols_to_drop = cols[:-1]
                    cols = list(df_vars_years.drop(cols_to_drop, axis = 1).columns)
                    df_me = pd.melt(df_vars_years
                                     , id_vars    = cols
                                     , var_name   = 'replicates'
                                     , value_name = 'replicate_weights')
                    df_me['sq_diff'] = (df_me['replicate_weights'] - df_me[weight])**2
                    df_me = df_me.groupby(cols, as_index = False)['sq_diff'].agg(sum)
                    df_me['variance'] = df_me['sq_diff']*(4/80)
                    df_me['SE'] = np.sqrt(df_me['variance'])
                    df_me['ME'] = df_me['SE']*1.645
                    df_me = df_me[cols + ['ME']].drop_duplicates()
                    df_vars_years = df_vars_years.drop(cols_to_drop, axis = 1)
                    df_vars_years = df_vars_years.drop_duplicates()
                    df_vars_years = df_vars_years.merge(df_me, on = cols, how = 'left')
                    df_vars_years = df_vars_years.drop_duplicates()
                list_df_years.append(df_vars_years)
                
                print('Success!')
                
            except Exception as e: print(e)
                                
    df_census_raw = pd.concat(list_df_years)


## For CPS tables
if estimate == 'CPS':

    
    print("Importing and compiling CPS data from the Census Bureau...")
    print("")
    
    list_df_census = []

    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                temp = query_census(df_urls      = df_urls
                                     , api_key   = api_key
                                     , estimate  = estimate
                                     , sample    = sample_type
                                     , geography = geography
                                     , variables = 'HRHHID,HRHHID2,'+','.join(dict_vars[str(year)])
                                     # , variables = ','.join(dict_vars[str(year)])
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])
                list_df_census.append(temp)
            except Exception as e: print(e)
                
    df_census_raw = pd.concat(list_df_census)

    # merge county name onto table
    df_census_raw['state' ] = df_census_raw['state' ].astype(str).apply('{:0>2}'.format)
    df_census_raw['county'] = df_census_raw['county'].astype(str).apply('{:0>3}'.format)
    df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                          , left_on = ['state', 'county']
                                          , right_on = ['State FIPS', 'County FIPS'])
    df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
    df_census_raw = df_census_raw.set_index(['state', 'county', 'County Name', 'Year']).reset_index()

print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")

Importing and compiling CPS data from the Census Bureau...

State: 06


100%|██████████| 14/14 [00:56<00:00,  4.03s/it]


Finished!!
Process complete.  It took --- 1.2 minutes ---


In [16]:
# view raw data
pd.set_option('display.max_columns', None)
print(df_census_raw.shape)
print(df_census_raw.Year.unique())
df_census_raw.head(3)

# For processing foodsec take min perrp by unique hrhhid and year

(9220, 14)
[2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022]


,state,county,County Name,Year,HRHHID,HRHHID2,HES1A,HES1B,HES1C,HES1D,PEHSPNON,PERRP,PTDTRACE,HHSUPWGT
0,06,061,Placer,2009,500164009202650,85002,1,2,1,2,2,1,1,4058.9581
1,06,061,Placer,2009,500164009202650,85002,1,2,1,2,2,4,1,4058.9581
2,06,061,Placer,2009,527091689600968,85001,2,2,1,1,2,2,1,3107.7496


***

## Processing

***

In [37]:
## Make copy of data frame
df_census = df_census_raw.copy()

# Replace weird missing values with np.nan
# Melt data from wide to long
# Convert imported values to numeric
# Merge cleam label field, variable mapping, race/ethnicity, and sorting field
# Remove unneeded columns
# Manually check column names and clean as needed
# Adjust dollars for inflation, if needed
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 1...')
    df_census = acs_processing_1(df_census, df_vars, indicator_name, geography)
    display(df_census.head(3))

# Reorganize margin of error fields
# Create "Categorical" race/ethnicity field for sorting
# Sort by geography, variable mapping, and race/ethnicity
# sort and then remove categorical field
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 2...')
    df_census = acs_processing_2(df_census, margin_of_error)
    display(df_census.head(3))

# Final processing step for ACS data
# Link various FIPS codes
# Roll up population/households/SE's to the desired geography and variable groupings
# Calculate percentages by geography, race/ethnicity, and variables
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 3...')
    if geography == 'Tracts':
        df_tracts1, df_tracts2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, df_fips)
        display(df_tracts1.head(3), df_tracts2.head(3))
    if geography == 'Counties':
        df_counties1, df_counties2, df_mpo1, df_mpo2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, df_fips)
        display(df_mpo1.head(3), df_mpo2.head(3))
    if geography == 'MSA':
        df_msa1, df_msa2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh)
        display(df_msa1.head(3), df_msa2.head(3))


if sample_type in ['PUMS', 'FOODSEC']:
    # Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
    # Convert weighted column to integer, convert value fields to string to use as merge field
    # Reshape data dictionary of values/descriptions and reorganize columns
    # Merge meaningful value descriptions onto imported data

    groups  = list(df_vars[df_vars['Data Type'].str.contains('group')]['ID2'].unique())
    groups2 = list(df_vars[df_vars['Data Type'] == 'group']['ID2'].unique())

    if sample_type == 'PUMS':
        df_census['PUMA'] = df_census['PUMA'].astype(str).apply('{:0>5}'.format)
        df_census[weight] = df_census[weight].astype(int)
        
    # if sample_type == 'FOODSEC':
    #     cols = ['state', 'county', 'County Name', 'Year'] + dict_vars[str(year)]
        # df_census = df_census[cols]

    for group in groups2:
        df_census[group] = df_census[group].astype(str).apply('{:0>2}'.format)
        
    df_vars   ['Value1'] = df_vars   ['Value1'].astype(str).apply('{:0>2}'.format)
    df_census ['state' ] = df_census ['state' ].astype(str).apply('{:0>2}'.format)

    if sample_type == 'PUMS':
        df_census[weight] = df_census[weight].astype(int)
    df_census[groups2] = df_census[groups2].astype("string")

    df_vars2 = df_vars.pivot_table(index = ['Year', 'Value1']
                                           , columns = 'ID2'
                                           , values = 'Description2'
                                           , aggfunc = lambda x: x).reset_index()
    cols = ['Year', 'Value1'] + groups2
    df_vars2 = df_vars2[cols]

    list_values = []
    for group in groups2:
        list_values = list_values + list(df_census[group].values)
    set_values = set(list_values)
    
    df_vars2 = df_vars2[df_vars2['Value1'].isin(set_values)]
    df_vars2 = df_vars2.add_suffix('_desc').rename(columns = {'Value1_desc':'Value1', 'Year_desc':'Year'})

    for col in cols[2:]:
        df_census = df_census.merge(df_vars2[['Value1', col+'_desc', 'Year']], left_on = [col, 'Year'], right_on = ['Value1', 'Year'], how = 'inner')
        df_census[col] = df_census[col+'_desc']
        df_census = df_census.drop(['Value1', col+'_desc'], axis = 1)

    # if 'HISP' in groups:
    #     df_census.loc[df_census['HISP'] == 'Hispanic or Latino', 'RAC1P'] = 'Hispanic or Latino'
    #     df_census = df_census.drop('HISP', axis = 1)
    #     groups.remove('HISP')

    # Need to condense race fields. If Hispanic, we categorize as Hispanic. If not, are they Black, Asian, Mixed, etc? Then we don't need the Hispanic/Not Hispanic column

    # if sample_type == 'FOODSEC':
    #     df_census.loc[df_census['PRDTHSP_desc'].str.contains(','), 'PRDTHSP_desc'] = 'Not Hispanic or Latino'
    display(df_census.head(3))


# Remove rows with missing values
# Sort by PUMA, Year, then by each group
# Only keep description mappings, remove the original PUMS values
# Rollup using suggested weight field
# Merge on PUMA name field
if sample_type == 'PUMS':
    df_census = df_census.dropna()
    
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums[df_fips_pums['STATEFP'].isin(list(dict_fips.keys()))]
    df_fips_pums = df_fips_pums[['STATEFP', 'PUMA5CE', 'PUMA NAME', 'COUNTYFP', 'Years']].rename(columns = {'PUMA5CE':'PUMA', 'STATEFP':'state'}).drop_duplicates()

    df_census1 = df_census[df_census['Year'].isin(sequence(2012, 2021, 1))]
    df_census2 = df_census[df_census['Year'].isin(sequence(2022, 2031, 1))]
    
    df_census1 = df_census1.merge(df_fips_pums[df_fips_pums['Years'] == '2012-2021'], on = ['state', 'PUMA'], how = 'left')
    df_census2 = df_census2.merge(df_fips_pums[df_fips_pums['Years'] == '2022-2031'], on = ['state', 'PUMA'], how = 'left')
    df_census = pd.concat([df_census1, df_census2])
    df_census = df_census.drop('Years', axis = 1)

    df_census['COUNTYFP'] = df_census['COUNTYFP'].astype(str).apply('{:0>3}'.format)
    df_census = df_census.merge(df_fips[['State FIPS', 'MPO', 'County FIPS', 'County Name', 'MSA_ID', 'MSA_acs']].drop_duplicates()
                                          , left_on = ['state', 'COUNTYFP']
                                          , right_on = ['State FIPS', 'County FIPS'])
    df_census.drop(['state', 'COUNTYFP'], axis = 1, inplace = True)
    df_census = df_census.rename(columns = {'MSA_acs':'MSA'})
    df_census = df_census.set_index(['State FIPS', 'MPO', 'MSA_ID', 'MSA', 'County FIPS', 'County Name', 'Year']).reset_index()
    df_census = df_census.sort_values(['State FIPS', 'PUMA', 'Year'] + groups, ascending = [True, True, False] + [item in groups for item in groups])
    
    if 'HISP' in groups:
        df_census.loc[df_census['HISP'] == 'Hispanic or Latino', 'RAC1P'] = 'Hispanic or Latino'
        df_census = df_census.drop('HISP', axis = 1)
        groups.remove('HISP')

    if 'HHLDRHISP' in groups:
        df_census.loc[df_census['HHLDRHISP'] == 'Hispanic or Latino', 'HHLDRRAC1P'] = 'Hispanic or Latino'
        df_census = df_census.drop('HHLDRHISP', axis = 1)

    if indicator_name == 'Cost_6':
        cols = ['GRPIP', 'OCPIP']
        df_census[cols] = df_census[cols].astype(int)
        conditions = [
                        ( (df_census['WGTP'] == 0) ),
                        ( (df_census['GRPIP'] == 0) & (df_census['OCPIP'] == 0) ),
                        ( (df_census['GRPIP'] == 0) & (df_census['OCPIP']  > 0) ),
                        ( (df_census['OCPIP'] == 0) & (df_census['GRPIP']  > 0) )
                    ]
        choices = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)', 'Owner', 'Renter']
        df_census["housing_type"] = np.select(conditions, choices)
        conditions = [
                        (  (df_census['WGTP'] ==  0) ),
                        (  (df_census['GRPIP'] ==  0) & (df_census['OCPIP'] == 0)),
                        ( ((df_census['GRPIP'] ==  0) & (df_census['OCPIP'] <= 30)) | ((df_census['OCPIP'] ==  0) & (df_census['GRPIP'] <= 30)) ),
                        ( ((df_census['GRPIP']  > 30) & (df_census['GRPIP'] <= 50)) | ((df_census['OCPIP']  > 30) & (df_census['OCPIP'] <= 50)) ),
                        (  (df_census['GRPIP']  > 50) | (df_census['OCPIP']  > 50) )
                    ]
        choices = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)', 'Cost burden <=30%', 'Cost burden >30% to <=50%', 'Cost burden >50%']
        df_census["housing_burden"] = np.select(conditions, choices)
        df_census = df_census.drop(['GRPIP', 'OCPIP'], axis = 1)
        groups = list(map(lambda x: x.replace('GRPIP', 'housing_type'  ), groups))
        groups = list(map(lambda x: x.replace('OCPIP', 'housing_burden'), groups))

    if indicator_name in ['Income_2', 'Accessibility_2', 'Accessibility_4']:
        df_cpi = pd.read_excel(os.path.join(path_config0, 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
        df_cpi = df_cpi[['Year', 'IAF_2023']]

        df_census = df_census.merge(df_cpi, on = 'Year', how = 'left')
        cols = ['HINCP', 'ADJINC']
        df_census[cols] = df_census[cols].astype('float32')
        df_census['HINCP'] = df_census['HINCP']*df_census['ADJINC']*df_census['IAF_2023']
        df_census = df_census.drop(['IAF_2023', 'ADJINC'], axis = 1)

        df_income_brackets = pd.read_excel(os.path.join(path_config0, 'CA State Income Brackets by Household Size.xlsx'), sheet_name = 'Table')
        df_income_brackets['County'].fillna(method='ffill', inplace = True)
        df_income_brackets['County'] = df_income_brackets['County'].str.replace(' County.*'         , '' , regex = True)
        df_income_brackets['County'] = df_income_brackets['County'].str.replace('\n'                , ' ', regex = True)
        df_income_brackets['AMI'   ] = df_income_brackets['County'].str.extract('\$?([0-9,]+)[.%]?')
        df_income_brackets['AMI'   ] = df_income_brackets['AMI'   ].str.replace(','                 , '' , regex = True)
        df_income_brackets['County'] = df_income_brackets['County'].str.replace(' \$?([0-9,]+)[.%]?', '' , regex = True)
        df_income_brackets = pd.melt(df_income_brackets
                                      , id_vars = ['County', 'Income Bracket', 'AMI']
                                      , var_name = 'NP'
                                      , value_name = 'Income Threshold')
        df_income_brackets = df_income_brackets[df_income_brackets['Income Bracket'].isin(['Low Income', 'Moderate Income'])]
        df_income_brackets = df_income_brackets.pivot_table(index = ['County', 'NP']
                                                             , columns = 'Income Bracket'
                                                             , values = 'Income Threshold').reset_index().rename(columns = {'County':'County Name'})
        df_income_brackets['NP'] = df_income_brackets['NP'].astype(str)
        df_census = df_census.merge(df_income_brackets, on = ['County Name', 'NP'], how = 'left')
        df_census.loc[ df_census['HINCP'] <= df_census['Low Income']                                                        , 'Income Bracket'] = 'Low Income'
        df_census.loc[(df_census['HINCP']  > df_census['Low Income']) & (df_census['HINCP'] <= df_census['Moderate Income']), 'Income Bracket'] = 'Moderate Income'
        df_census.loc[ df_census['HINCP']  > df_census['Moderate Income']                                                   , 'Income Bracket'] = 'High Income'
        df_census.loc[ df_census['NP'] == 0                                                                                 , 'Income Bracket'] = 'No data available'
        df_census = df_census.drop(['HINCP', 'NP'], axis = 1)
        groups = ['Income Bracket'] + groups[:-1]
        if indicator_name == 'Accessibility_4':
            df_census['JWMNP'] = df_census['JWMNP'].astype('float32')
            df_census.loc[(df_census['JWMNP'] ==  0)                             , 'Travel Time'] = 'No commute (worked from home)'
            df_census.loc[(df_census['JWMNP']  >  0) & (df_census['JWMNP'] <= 15), 'Travel Time'] = '0 to 15 minutes'
            df_census.loc[(df_census['JWMNP']  > 15) & (df_census['JWMNP'] <= 30), 'Travel Time'] = '15 to 30 minutes'
            df_census.loc[(df_census['JWMNP']  > 30)                             , 'Travel Time'] = 'More than 30 minutes'
            groups = groups + ['Travel Time']

    df_puma     = df_census.drop([                            'MSA_ID', 'MSA', 'County FIPS', 'County Name', 'SERIALNO'], axis = 1)
    df_counties = df_census.drop([       'PUMA', 'PUMA NAME', 'MSA_ID', 'MSA',                               'SERIALNO'], axis = 1)
    df_msa      = df_census.drop(['MPO', 'PUMA', 'PUMA NAME',                  'County FIPS', 'County Name', 'SERIALNO'], axis = 1)
    df_mpo      = df_census.drop([       'PUMA', 'PUMA NAME', 'MSA_ID', 'MSA', 'County FIPS', 'County Name', 'SERIALNO'], axis = 1)

    group_puma     = ['State FIPS', 'MPO', 'PUMA'       , 'PUMA NAME'  ]
    group_counties = ['State FIPS', 'MPO', 'County FIPS', 'County Name']
    group_msa      = ['State FIPS',        'MSA_ID'     , 'MSA'        ]
    group_mpo      = ['State FIPS', 'MPO'                              ]
    
    df_puma     = df_puma    .set_index(group_puma    ).reset_index()
    df_counties = df_counties.set_index(group_counties).reset_index()
    df_msa      = df_msa     .set_index(group_msa     ).reset_index()
    df_mpo      = df_mpo     .set_index(group_mpo     ).reset_index()

    if margin_of_error == 'Yes':
        df_puma    .loc[df_puma    ['ME'] < 0, 'ME'] = np.nan
        df_counties.loc[df_counties['ME'] < 0, 'ME'] = np.nan
        df_msa     .loc[df_msa     ['ME'] < 0, 'ME'] = np.nan
        df_mpo     .loc[df_mpo     ['ME'] < 0, 'ME'] = np.nan
        
        if indicator_name in ['Income_2', 'Accessibility_2', 'Accessibility_4']:
            if indicator_name == 'Accessibility_4':
                df_puma1     = df_puma    .groupby(group_puma     + ['Year', 'RAC1P', 'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_counties1 = df_counties.groupby(group_counties + ['Year', 'RAC1P', 'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_msa1      = df_msa     .groupby(group_msa      + ['Year', 'RAC1P', 'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_mpo1      = df_mpo     .groupby(group_mpo      + ['Year', 'RAC1P', 'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_puma2     = df_puma    .groupby(group_puma     + ['Year',          'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_counties2 = df_counties.groupby(group_counties + ['Year',          'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_msa2      = df_msa     .groupby(group_msa      + ['Year',          'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_mpo2      = df_mpo     .groupby(group_mpo      + ['Year',          'Income Bracket', 'Travel Time'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_puma2    .loc[:, 'RAC1P'] = 'All'
                df_counties2.loc[:, 'RAC1P'] = 'All'
                df_msa2     .loc[:, 'RAC1P'] = 'All'
                df_mpo2     .loc[:, 'RAC1P'] = 'All'
                df_puma     = pd.concat([df_puma1    , df_puma2    ])
                df_counties = pd.concat([df_counties1, df_counties2])
                df_msa      = pd.concat([df_msa1     , df_msa2     ])
                df_mpo      = pd.concat([df_mpo1     , df_mpo2     ])
                
            if indicator_name == 'Accessibility_2':
                df_puma     = df_puma    .groupby(group_puma     + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_counties = df_counties.groupby(group_counties + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_msa      = df_msa     .groupby(group_msa      + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_mpo      = df_mpo     .groupby(group_mpo      + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            
            if indicator_name == 'Income_2':
                df_puma     = df_puma    .groupby(group_puma     + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_counties = df_counties.groupby(group_counties + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_msa      = df_msa     .groupby(group_msa      + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
                df_mpo      = df_mpo     .groupby(group_mpo      + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))


        elif indicator_name == 'Cost_6':
            df_puma1 = df_puma.groupby(list(df_puma.drop([weight, 'ME'         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_puma2 = df_puma.groupby(list(df_puma.drop([weight, 'ME', 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_puma2.loc[:, 'RAC1P'] = 'All'
            df_puma2 = pd.concat([df_puma1, df_puma2])
            df_puma3 = df_puma2[df_puma2['housing_type'].isin(['Owner', 'Renter'])]
            df_puma3 = df_puma3.groupby(list(df_puma3.drop(['Total', 'ME', 'housing_type'], axis = 1).columns), as_index = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
            df_puma3.loc[:, 'housing_type'] = 'Renters/Owners'
            df_puma = pd.concat([df_puma2, df_puma3])

            df_counties1 = df_counties.groupby(list(df_counties.drop([weight, 'ME'         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_counties2 = df_counties.groupby(list(df_counties.drop([weight, 'ME', 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_counties2.loc[:, 'RAC1P'] = 'All'
            df_counties2 = pd.concat([df_counties1, df_counties2])
            df_counties3 = df_counties2[df_counties2['housing_type'].isin(['Owner', 'Renter'])]
            df_counties3 = df_counties3.groupby(list(df_counties3.drop(['Total', 'ME', 'housing_type'], axis = 1).columns), as_index = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
            df_counties3.loc[:, 'housing_type'] = 'Renters/Owners'
            df_counties = pd.concat([df_counties2, df_counties3])

            df_msa1 = df_msa.groupby(list(df_msa.drop([weight, 'ME'         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_msa2 = df_msa.groupby(list(df_msa.drop([weight, 'ME', 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_msa2.loc[:, 'RAC1P'] = 'All'
            df_msa2 = pd.concat([df_msa1, df_msa2])
            df_msa3 = df_msa2[df_msa2['housing_type'].isin(['Owner', 'Renter'])]
            df_msa3 = df_msa3.groupby(list(df_msa3.drop(['Total', 'ME', 'housing_type'], axis = 1).columns), as_index = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
            df_msa3.loc[:, 'housing_type'] = 'Renters/Owners'
            df_msa = pd.concat([df_msa2, df_msa3])

            df_mpo1 = df_mpo.groupby(list(df_mpo.drop([weight, 'ME'         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_mpo2 = df_mpo.groupby(list(df_mpo.drop([weight, 'ME', 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_mpo2.loc[:, 'RAC1P'] = 'All'
            df_mpo2 = pd.concat([df_mpo1, df_mpo2])
            df_mpo3 = df_mpo2[df_mpo2['housing_type'].isin(['Owner', 'Renter'])]
            df_mpo3 = df_mpo3.groupby(list(df_mpo3.drop(['Total', 'ME', 'housing_type'], axis = 1).columns), as_index = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
            df_mpo3.loc[:, 'housing_type'] = 'Renters/Owners'
            df_mpo = pd.concat([df_mpo2, df_mpo3])

            df_puma    ['ME_ratio'] = df_puma    ['ME']/df_puma    ['Total']*100
            df_counties['ME_ratio'] = df_counties['ME']/df_counties['Total']*100
            df_msa     ['ME_ratio'] = df_msa     ['ME']/df_msa     ['Total']*100
            df_mpo     ['ME_ratio'] = df_mpo     ['ME']/df_mpo     ['Total']*100
        
        else:
            df_puma     = df_puma    .groupby(list(df_puma    .drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_counties = df_counties.groupby(list(df_counties.drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_msa      = df_msa     .groupby(list(df_msa     .drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
            df_mpo      = df_mpo     .groupby(list(df_mpo     .drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'), ME = ('ME', sqrtsumsq))
        
        df_puma    ['ME_ratio'] = df_puma    ['ME']/df_puma    ['Total']*100
        df_counties['ME_ratio'] = df_counties['ME']/df_counties['Total']*100
        df_msa     ['ME_ratio'] = df_msa     ['ME']/df_msa     ['Total']*100
        df_mpo     ['ME_ratio'] = df_mpo     ['ME']/df_mpo     ['Total']*100
        
        conditions = [df_puma['ME_ratio'] <= MOE_thresh, df_puma['ME_ratio']  > MOE_thresh]
        choices = ['Yes', 'No']
        df_puma['Use for Reporting'] = np.select(conditions, choices, default = 'No')
        
        conditions = [df_counties['ME_ratio'] <= MOE_thresh, df_counties['ME_ratio']  > MOE_thresh]
        choices = ['Yes', 'No']
        df_counties['Use for Reporting'] = np.select(conditions, choices, default = 'No')
        
        conditions = [df_msa['ME_ratio'] <= MOE_thresh, df_msa['ME_ratio']  > MOE_thresh]
        choices = ['Yes', 'No']
        df_msa['Use for Reporting'] = np.select(conditions, choices, default = 'No')
        
        conditions = [df_mpo['ME_ratio'] <= MOE_thresh, df_mpo['ME_ratio']  > MOE_thresh]
        choices = ['Yes', 'No']
        df_mpo['Use for Reporting'] = np.select(conditions, choices, default = 'No')
    
    if margin_of_error == 'No':
        if indicator_name in ['Income_2', 'Accessibility_2']:
            if indicator_name == 'Accessibility_2':
                df_puma     = df_puma    .groupby(group_puma     + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'))
                df_counties = df_counties.groupby(group_counties + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'))
                df_msa      = df_msa     .groupby(group_msa      + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'))
                df_mpo      = df_mpo     .groupby(group_mpo      + ['Year', 'Income Bracket', 'JWTRNS'], as_index = False).agg(Total = (weight, 'sum'))
            
            if indicator_name == 'Income_2':
                df_puma     = df_puma    .groupby(group_puma     + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'))
                df_counties = df_counties.groupby(group_counties + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'))
                df_msa      = df_msa     .groupby(group_msa      + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'))
                df_mpo      = df_mpo     .groupby(group_mpo      + ['Year', 'Income Bracket'], as_index = False).agg(Total = (weight, 'sum'))
                
        if indicator_name == 'Cost_6':
            df_puma1 = df_puma.groupby(list(df_puma.drop([weight         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_puma2 = df_puma.groupby(list(df_puma.drop([weight, 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_puma2.loc[:, 'RAC1P'] = 'All'
            df_puma2 = pd.concat([df_puma1, df_puma2])
            df_puma3 = df_puma2[df_puma2['housing_type'].isin(['Owner', 'Renter'])]
            df_puma3 = df_puma3.groupby(list(df_puma3.drop(['Total'], axis = 1).columns), as_index = False).agg(Total = ('Total', 'sum'))
            df_puma3.loc[:, 'housing_type'] = 'Renters and Owners'
            df_puma3['Percentage'] = 100*df_puma3['Total']/df_puma3.groupby(list(df_puma3.drop(['housing_burden', 'Total'], axis = 1).columns))['Total'].transform('sum')
            df_puma = pd.concat([df_puma2, df_puma3])

            df_counties1 = df_counties.groupby(list(df_counties.drop([weight         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_counties2 = df_counties.groupby(list(df_counties.drop([weight, 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_counties2.loc[:, 'RAC1P'] = 'All'
            df_counties2 = pd.concat([df_counties1, df_counties2])
            df_counties3 = df_counties2[df_counties2['housing_type'].isin(['Owner', 'Renter'])]
            df_counties3 = df_counties3.groupby(list(df_counties3.drop(['Total'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_counties3.loc[:, 'housing_type'] = 'Renters and Owners'
            df_counties = pd.concat([df_counties2, df_counties3])

            df_msa1 = df_msa.groupby(list(df_msa.drop([weight         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_msa2 = df_msa.groupby(list(df_msa.drop([weight, 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_msa2.loc[:, 'RAC1P'] = 'All'
            df_msa2 = pd.concat([df_msa1, df_msa2])
            df_msa3 = df_msa2[df_msa2['housing_type'].isin(['Owner', 'Renter'])]
            df_msa3 = df_msa3.groupby(list(df_msa3.drop(['Total'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_msa3.loc[:, 'housing_type'] = 'Renters and Owners'
            df_msa = pd.concat([df_msa2, df_msa3])
            
            df_mpo1 = df_mpo.groupby(list(df_mpo.drop([weight         ], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_mpo2 = df_mpo.groupby(list(df_mpo.drop([weight, 'RAC1P'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_mpo2.loc[:, 'RAC1P'] = 'All'
            df_mpo2 = pd.concat([df_mpo1, df_mpo2])
            df_mpo3 = df_mpo2[df_mpo2['housing_type'].isin(['Owner', 'Renter'])]
            df_mpo3 = df_mpo3.groupby(list(df_mpo3.drop(['Total'], axis = 1).columns), as_index = False).agg(Total = (weight, 'sum'))
            df_mpo3.loc[:, 'housing_type'] = 'Renters and Owners'
            df_mpo = pd.concat([df_mpo2, df_mpo3])

        else:
            df_puma     = df_puma    .groupby(list(df_puma    .drop([weight], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'))
            df_counties = df_counties.groupby(list(df_counties.drop([weight], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'))
            df_msa      = df_msa     .groupby(list(df_msa     .drop([weight], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'))
            df_mpo      = df_mpo     .groupby(list(df_mpo     .drop([weight], axis = 1).columns), as_index = False, sort = False).agg(Total = (weight, 'sum'))
    
    if percentages == 'Yes':
        if margin_of_error == 'Yes':
            if len(groups) > 1:
                df_puma    ['Percentage'] = 100*df_puma    ['Total'] / df_puma    .groupby(list(df_puma    .drop(groups[-1:] + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
                df_counties['Percentage'] = 100*df_counties['Total'] / df_counties.groupby(list(df_counties.drop(groups[-1:] + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
                df_msa     ['Percentage'] = 100*df_msa     ['Total'] / df_msa     .groupby(list(df_msa     .drop(groups[-1:] + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
                df_mpo     ['Percentage'] = 100*df_mpo     ['Total'] / df_mpo     .groupby(list(df_mpo     .drop(groups[-1:] + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
            if len(groups) == 1:
                df_puma    ['Percentage'] = 100*df_puma    ['Total'] / df_puma    .groupby(list(df_puma    .drop(groups      + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
                df_counties['Percentage'] = 100*df_counties['Total'] / df_counties.groupby(list(df_counties.drop(groups      + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
                df_msa     ['Percentage'] = 100*df_msa     ['Total'] / df_msa     .groupby(list(df_msa     .drop(groups      + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
                df_mpo     ['Percentage'] = 100*df_mpo     ['Total'] / df_mpo     .groupby(list(df_mpo     .drop(groups      + ['Total', 'ME', 'ME_ratio', 'Use for Reporting'], axis = 1).columns))['Total'].transform('sum')
        if margin_of_error == 'No':
            if len(groups) > 1:
                df_puma    ['Percentage'] = 100*df_puma    ['Total'] / df_puma    .groupby(list(df_puma    .drop(groups[-1:] + ['Total'], axis = 1).columns))['Total'].transform('sum')
                df_counties['Percentage'] = 100*df_counties['Total'] / df_counties.groupby(list(df_counties.drop(groups[-1:] + ['Total'], axis = 1).columns))['Total'].transform('sum')
                df_msa     ['Percentage'] = 100*df_msa     ['Total'] / df_msa     .groupby(list(df_msa     .drop(groups[-1:] + ['Total'], axis = 1).columns))['Total'].transform('sum')
                df_mpo     ['Percentage'] = 100*df_mpo     ['Total'] / df_mpo     .groupby(list(df_mpo     .drop(groups[-1:] + ['Total'], axis = 1).columns))['Total'].transform('sum')
            if len(groups) == 1:
                df_puma    ['Percentage'] = 100*df_puma    ['Total'] / df_puma    .groupby(list(df_puma    .drop(groups      + ['Total'], axis = 1).columns))['Total'].transform('sum')
                df_counties['Percentage'] = 100*df_counties['Total'] / df_counties.groupby(list(df_counties.drop(groups      + ['Total'], axis = 1).columns))['Total'].transform('sum')
                df_msa     ['Percentage'] = 100*df_msa     ['Total'] / df_msa     .groupby(list(df_mpo     .drop(groups      + ['Total'], axis = 1).columns))['Total'].transform('sum')                
                df_mpo     ['Percentage'] = 100*df_mpo     ['Total'] / df_mpo     .groupby(list(df_mpo     .drop(groups      + ['Total'], axis = 1).columns))['Total'].transform('sum')                
            

    display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))
    

<>:179: SyntaxWarning: invalid escape sequence '\$'
<>:181: SyntaxWarning: invalid escape sequence '\$'
<>:179: SyntaxWarning: invalid escape sequence '\$'
<>:181: SyntaxWarning: invalid escape sequence '\$'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_34024\2806768206.py:179: SyntaxWarning: invalid escape sequence '\$'
  df_income_brackets['AMI'   ] = df_income_brackets['County'].str.extract('\$?([0-9,]+)[.%]?')
C:\Users\jchoy\AppData\Local\Temp\ipykernel_34024\2806768206.py:181: SyntaxWarning: invalid escape sequence '\$'
  df_income_brackets['County'] = df_income_brackets['County'].str.replace(' \$?([0-9,]+)[.%]?', '' , regex = True)
C:\Users\jchoy\AppData\Local\Temp\ipykernel_34024\2806768206.py:179: SyntaxWarning: invalid escape sequence '\$'
  df_income_brackets['AMI'   ] = df_income_brackets['County'].str.extract('\$?([0-9,]+)[.%]?')
C:\Users\jchoy\AppData\Local\Temp\ipykernel_34024\2806768206.py:181: SyntaxWarning: invalid escape sequence '\$'
  df_income_brackets['County'] = df

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [23]:
if sample_type in ['PUMS', 'FOODSEC']:
    # Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
    # Convert weighted column to integer, convert value fields to string to use as merge field
    # Reshape data dictionary of values/descriptions and reorganize columns
    # Merge meaningful value descriptions onto imported data

    groups  = list(df_vars[df_vars['Data Type'].str.contains('group', na=False)]['ID2'].unique())
    groups2 = list(df_vars[df_vars['Data Type'] == 'group']['ID2'].unique())

    if sample_type == 'PUMS':
        df_census['PUMA'] = df_census['PUMA'].astype(str).apply('{:0>5}'.format)
        df_census[weight] = df_census[weight].astype(int)
        
    # if sample_type == 'FOODSEC':
    #     cols = ['state', 'county', 'County Name', 'Year'] + dict_vars[str(year)]
        # df_census = df_census[cols]

    for group in groups2:
        df_census[group] = df_census[group].astype(str).apply('{:0>2}'.format)
        
    df_vars   ['Value1'] = df_vars   ['Value1'].astype(str).apply('{:0>2}'.format)
    df_census ['state' ] = df_census ['state' ].astype(str).apply('{:0>2}'.format)

    if sample_type == 'PUMS':
        df_census[weight] = df_census[weight].astype(int)
    df_census[groups2] = df_census[groups2].astype("string")

    df_vars2 = df_vars.pivot_table(index = ['Year', 'Value1']
                                           , columns = 'ID2'
                                           , values = 'Description2'
                                           , aggfunc = lambda x: x).reset_index()
    cols = ['Year', 'Value1'] + groups2
    df_vars2 = df_vars2[cols]

    list_values = []
    for group in groups2:
        list_values = list_values + list(df_census[group].values)
    set_values = set(list_values)
    
    df_vars2 = df_vars2[df_vars2['Value1'].isin(set_values)]
    df_vars2 = df_vars2.add_suffix('_desc').rename(columns = {'Value1_desc':'Value1', 'Year_desc':'Year'})

    for col in cols[2:]:
        df_census = df_census.merge(df_vars2[['Value1', col+'_desc', 'Year']], left_on = [col, 'Year'], right_on = ['Value1', 'Year'], how = 'inner')
        df_census[col] = df_census[col+'_desc']
        df_census = df_census.drop(['Value1', col+'_desc'], axis = 1)

    # if 'HISP' in groups:
    #     df_census.loc[df_census['HISP'] == 'Hispanic or Latino', 'RAC1P'] = 'Hispanic or Latino'
    #     df_census = df_census.drop('HISP', axis = 1)
    #     groups.remove('HISP')

    # Need to condense race fields. If Hispanic, we categorize as Hispanic. If not, are they Black, Asian, Mixed, etc? Then we don't need the Hispanic/Not Hispanic column

    # if sample_type == 'FOODSEC':
    #     df_census.loc[df_census['PRDTHSP_desc'].str.contains(','), 'PRDTHSP_desc'] = 'Not Hispanic or Latino'
    display(df_census.head(3))


,state,county,County Name,Year,HRHHID,HRHHID2,HES1A,HES1B,HES1C,HES1D,PEHSPNON,PERRP,PTDTRACE,HHSUPWGT
0,06,061,Placer,2009,500164009202650,85002,Yes,No,Yes,No,Non-Hispanic,1,White (NH),4058.9581
1,06,061,Placer,2009,500164009202650,85002,Yes,No,Yes,No,Non-Hispanic,4,White (NH),4058.9581
2,06,061,Placer,2009,527091689600968,85001,No,No,Yes,Yes,Non-Hispanic,2,White (NH),3107.7496


In [40]:
df_census['HHID'] = df_census['HRHHID'] + df_census['HRHHID2']

In [43]:
idx_b = df_census.groupby(['Year', 'HHID'])['PERRP'].idxmin()

# Use the indices to filter the original dataframe
test = df_census.loc[idx_b].reset_index(drop=True)

In [50]:
len(test['Year'].unique())

14

In [45]:
test

,state,county,County Name,Year,HRHHID,HRHHID2,HES1A,HES1B,HES1C,HES1D,PEHSPNON,PERRP,PTDTRACE,HHSUPWGT,HHID
0,06,113,Yolo,2009,000075204047661,85001,2,2,2,2,2,2,7,3427.3389,00007520404766185001
1,06,067,Sacramento,2009,000563096604389,85001,1,2,1,2,2,1,1,3380.6822,00056309660438985001
2,06,067,Sacramento,2009,000663096608389,86001,1,1,1,2,2,1,1,4044.0582,00066309660838986001
3,06,061,Placer,2009,002075806016621,85001,2,2,2,2,1,1,1,4622.3066,00207580601662185001
4,06,067,Sacramento,2009,003905360695986,85001,-1,-1,-1,-1,2,1,1,0.0000,00390536069598685001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3552,06,067,Sacramento,2022,910101110251676,13011,1,2,2,2,2,40,1,4715.5292,91010111025167613011
3553,06,067,Sacramento,2022,910101110811676,15011,-1,-1,-1,-1,2,40,1,0.0000,91010111081167615011
3554,06,067,Sacramento,2022,910109360018676,15011,1,1,1,2,2,40,4,6411.5418,91010936001867615011
3555,06,067,Sacramento,2022,910109360128676,13011,2,1,-2,2,2,40,1,8963.5011,91010936012867613011


In [49]:
len(test['HHID'].unique())

2242

In [31]:
df_census[df_census['HRHHID2'] == '85001']

,state,county,County Name,Year,HRHHID,HRHHID2,HES1A,HES1B,HES1C,HES1D,PEHSPNON,PERRP,PTDTRACE,HHSUPWGT
2,06,061,Placer,2009,527091689600968,85001,No,No,Yes,Yes,Non-Hispanic,2,White (NH),3107.7496
3,06,061,Placer,2009,850966099503599,85001,Yes,Yes,Yes,No,Non-Hispanic,2,White (NH),3294.2847
4,06,061,Placer,2009,065950360995899,85001,Yes,Yes,Yes,No,Non-Hispanic,1,White (NH),3623.5640
5,06,061,Placer,2009,065950360995899,85001,Yes,Yes,Yes,No,Non-Hispanic,3,White (NH),3623.5640
6,06,061,Placer,2009,065950360995899,85001,Yes,Yes,Yes,No,Non-Hispanic,4,White (NH),3623.5640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,06,067,Sacramento,2009,439003680600196,85001,Yes,Yes,Yes,No,Non-Hispanic,3,Two or more races (NH),4086.6223
612,06,067,Sacramento,2009,439003680600196,85001,Yes,Yes,Yes,No,Non-Hispanic,4,Two or more races (NH),4086.6223
613,06,067,Sacramento,2009,439003680600196,85001,Yes,Yes,Yes,No,Non-Hispanic,4,Two or more races (NH),4086.6223
614,06,067,Sacramento,2009,439003680600196,85001,Yes,Yes,Yes,No,Non-Hispanic,4,Two or more races (NH),4086.6223


In [32]:
idx = df_census.groupby(['Year', 'HRHHID2'])['PERRP'].idxmin()

# Use the indices to filter the original dataframe
df_min_perrp = df_census.loc[idx].reset_index(drop=True)

In [33]:
df_min_perrp

,state,county,County Name,Year,HRHHID,HRHHID2,HES1A,HES1B,HES1C,HES1D,PEHSPNON,PERRP,PTDTRACE,HHSUPWGT
0,06,061,Placer,2009,065950360995899,85001,Yes,Yes,Yes,No,Non-Hispanic,1,White (NH),3623.5640
1,06,061,Placer,2009,500164009202650,85002,Yes,No,Yes,No,Non-Hispanic,1,White (NH),4058.9581
2,06,061,Placer,2009,826209840679001,85261,Yes,No,No,No,Non-Hispanic,2,White (NH),3858.0636
3,06,067,Sacramento,2009,044006119060450,86001,Yes,Yes,Yes,No,Non-Hispanic,1,White (NH),3184.7618
4,06,061,Placer,2009,069630160998999,87001,Yes,Yes,Yes,No,Non-Hispanic,1,White (NH),3779.2669
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,06,061,Placer,2022,026205801670001,13012,Yes,Yes,Yes,No,Non-Hispanic,40,White (NH),5696.0758
72,06,067,Sacramento,2022,013400610068639,13013,Yes,No,Yes,No,Non-Hispanic,41,Asian (NH),3725.3831
73,06,061,Placer,2022,000610296585569,14011,Yes,Yes,No,No,Non-Hispanic,40,White (NH),4944.9084
74,06,067,Sacramento,2022,616670119111000,14012,Yes,No,Yes,No,Non-Hispanic,41,Black or African American (NH),9006.7587


In [11]:
# Final renaming of tables for cleanliness
if geography == 'Tracts':
    df_tracts1 = rename_census(df_tracts1        = df_tracts1
                               , geography       = geography
                               , indicator_name  = indicator_name
                               , margin_of_error = margin_of_error)
    display(df_tracts1.head(3))
if geography == 'Counties':
    df_counties1, df_mpo1 = rename_census(df_counties1      = df_counties1
                                          , df_mpo1         = df_mpo1
                                          , geography       = geography
                                          , indicator_name  = indicator_name
                                          , margin_of_error = margin_of_error)
    display(df_counties1.head(3), df_mpo1.head(3))
if geography == 'MSA':
    df_msa1 = rename_census(df_msa1           = df_msa1
                            , geography       = geography
                            , indicator_name  = indicator_name
                            , margin_of_error = margin_of_error)
    display(df_msa1.head(3))
if geography == 'PUMA':
    df_puma, df_counties, df_msa, df_mpo = rename_census(df_puma           = df_puma
                                                         , df_counties     = df_counties
                                                         , df_msa          = df_msa
                                                         , df_mpo          = df_mpo
                                                         , geography       = geography
                                                         , indicator_name  = indicator_name
                                                         , margin_of_error = margin_of_error
                                                         , groups          = groups)
    display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))

,State FIPS,MPO,PUMA,PUMA NAME,Year,VEH,Households,Percentage,Margin of Error,Margin of Error Ratio,Use for Reporting
4,06,SACOG,01700,El Dorado County--El Dorado Hills,2021,1 vehicle,17955,23.969082,2830.802278,15.766095,No
5,06,SACOG,01700,El Dorado County--El Dorado Hills,2021,2 vehicles,27882,37.221162,3295.643596,11.819968,No
6,06,SACOG,01700,El Dorado County--El Dorado Hills,2021,3 or more vehicles,26387,35.225407,3231.150041,12.245235,No


,State FIPS,MPO,County FIPS,County Name,Year,VEH,Households,Percentage,Margin of Error,Margin of Error Ratio,Use for Reporting
0,06,SACOG,017,El Dorado,2022,1 vehicle,16226,21.816177,2449.279937,15.094786,No
1,06,SACOG,017,El Dorado,2022,2 vehicles,28311,38.064698,3335.948932,11.783225,No
2,06,SACOG,017,El Dorado,2022,3 or more vehicles,27609,37.120845,3216.214126,11.649151,No


,State FIPS,MSA_ID,MSA,Year,VEH,Households,Percentage,Margin of Error,Margin of Error Ratio,Use for Reporting
0,06,40900,"Sacramento-Roseville-Folsom, CA Metro Area",2022,1 vehicle,269046,30.412843,10516.628432,3.908859,Yes
1,06,40900,"Sacramento-Roseville-Folsom, CA Metro Area",2022,2 vehicles,334535,37.815691,11340.317400,3.389875,Yes
2,06,40900,"Sacramento-Roseville-Folsom, CA Metro Area",2022,3 or more vehicles,231021,26.114514,9548.363461,4.133115,Yes


,State FIPS,MPO,Year,VEH,Households,Percentage,Margin of Error,Margin of Error Ratio,Use for Reporting
0,06,SACOG,2022,1 vehicle,298666,29.542519,11126.331744,3.725343,Yes
1,06,SACOG,2022,2 vehicles,380943,37.680940,12123.128789,3.182400,Yes
2,06,SACOG,2022,3 or more vehicles,274997,27.201302,10456.535742,3.802418,Yes


***

## Exporting

***

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )

if geography == 'Tracts':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Tracts '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_tracts1.to_excel(writer, index = False, sheet_name = 'Tracts')
        # df_tracts2.to_excel(writer, index = False, sheet_name = 'Tracts wide')

if geography == 'Counties':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
        # df_counties2.to_excel(writer, index = False, sheet_name = 'Counties wide')
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_mpo1.to_excel(writer, index = False, sheet_name = 'MPO')
        # df_mpo2.to_excel(writer, index = False, sheet_name = 'MPO wide')
             

if geography == 'MSA':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_msa1.to_excel(writer, index = False, sheet_name = 'MSA')
        # df_msa2.to_excel(writer, index = False, sheet_name = 'MSA wide')


if geography == 'PUMA':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' PUMA '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
    #     df_puma.to_excel(writer, index = False, sheet_name = 'PUMA')
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        df_counties.to_excel(writer, index = False, sheet_name = 'Counties')
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
    #     df_msa.to_excel(writer, index = False, sheet_name = 'MSA')
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        df_mpo.to_excel(writer, index = False, sheet_name = 'MPO')


print('')
print("Successfully exported")